# Selection Miss and Shell Usage

This notebook separates failed gold-agent selection from successful off-gold solution paths. `Selection Miss` is the pipeline label for a failed episode that discovered, but did not execute, every gold agent. `Usage Off-Gold` is a successful episode that did not execute every gold agent.

All selection-path metrics use the main Adaptive System test folds. Episodes without annotated gold agents are excluded from coverage metrics.


In [1]:
from pathlib import Path
import sys

import pandas as pd

analysis_dir = Path.cwd() / "analysis"
if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

from metrics import agents, paths
from metrics.conditions import GAIA_CONFIG, OFFICEBENCH_CONFIG

CASES = [
    ("rich", OFFICEBENCH_CONFIG),
    ("sparse", OFFICEBENCH_CONFIG),
    ("rich", GAIA_CONFIG),
    ("sparse", GAIA_CONFIG),
]


## Failed Selection


In [2]:
selection_misses = pd.concat(
    [paths.selection_miss_table(card, config) for card, config in CASES],
    ignore_index=True,
).set_index(["Benchmark", "Cards"])
display(selection_misses)


Selection Misses  Episodes  Selection Miss (%)
Benchmark   Cards                                                 
OfficeBench Rich                  95       542                17.5
            Sparse               128       542                23.6
GAIA        Rich                  40       300                13.3
            Sparse                45       300                15.0

## Successful Off-Gold Paths

The all-episode rate reconciles AllGold with the success-first pipeline. The success-conditional rate describes how often successful behavior departs from the annotated gold route.


In [3]:
off_gold_success = pd.concat(
    [paths.off_gold_success_table(card, config) for card, config in CASES],
    ignore_index=True,
).set_index(["Benchmark", "Cards", "Metric"])
display(off_gold_success)


Off-Gold Successes  \
Benchmark   Cards  Metric                                   
OfficeBench Rich   Retrieval Off-Gold                  57   
                   Usage Off-Gold                     131   
            Sparse Retrieval Off-Gold                  44   
                   Usage Off-Gold                     113   
GAIA        Rich   Retrieval Off-Gold                   0   
                   Usage Off-Gold                      12   
            Sparse Retrieval Off-Gold                   3   
                   Usage Off-Gold                      18   

                                       Gold-Annotated Episodes  \
Benchmark   Cards  Metric                                        
OfficeBench Rich   Retrieval Off-Gold                      542   
                   Usage Off-Gold                          542   
            Sparse Retrieval Off-Gold                      542   
                   Usage Off-Gold                          542   
GAIA        Rich   Retrieval Off-Gold                      277   
                   Usage Off-Gold                          277   
            Sparse Retrieval Off-Gold                      277   
                   Usage Off-Gold                          277   

                                       Successful Episodes  All Episodes (%)  \
Benchmark   Cards  Metric                                                      
OfficeBench Rich   Retrieval Off-Gold                  268              10.5   
                   Usage Off-Gold                      268              24.2   
            Sparse Retrieval Off-Gold                  241               8.1   
                   Usage Off-Gold                      241              20.8   
GAIA        Rich   Retrieval Off-Gold                   86               0.0   
                   Usage Off-Gold                       86               4.3   
            Sparse Retrieval Off-Gold                   82               1.1   
                   Usage Off-Gold                       82               6.5   

                                       Among Successes (%)  
Benchmark   Cards  Metric                                   
OfficeBench Rich   Retrieval Off-Gold                 21.3  
                   Usage Off-Gold                     48.9  
            Sparse Retrieval Off-Gold                 18.3  
                   Usage Off-Gold                     46.9  
GAIA        Rich   Retrieval Off-Gold                  0.0  
                   Usage Off-Gold                     14.0  
            Sparse Retrieval Off-Gold                  3.7  
                   Usage Off-Gold                     22.0

## OfficeBench Shell Usage

Shell usage is episode-level because `executed_agents` is deduplicated. A Shell workaround candidate is successful, uses Shell outside the gold set, and omits at least one gold agent; it is an association rather than proof of causal substitution.


In [5]:
shell_usage = pd.concat(
    [paths.shell_workaround_table(card) for card in ("rich", "sparse")],
    ignore_index=True,
).set_index(["Cards", "Metric", "Population"])
display(shell_usage.drop(columns="Benchmark"))


Count  \
Cards  Metric                     Population                               
Rich   Shell used                 All gold-annotated episodes        422   
                                  Successful episodes                204   
       Non-gold Shell used        Successful episodes                120   
       Shell workaround candidate Successful episodes                 54   
                                  Successful episodes using Shell     54   
Sparse Shell used                 All gold-annotated episodes        371   
                                  Successful episodes                166   
       Non-gold Shell used        Successful episodes                 93   
       Shell workaround candidate Successful episodes                 42   
                                  Successful episodes using Shell     42   

                                                                   Denominator  \
Cards  Metric                     Population                                     
Rich   Shell used                 All gold-annotated episodes              542   
                                  Successful episodes                      268   
       Non-gold Shell used        Successful episodes                      268   
       Shell workaround candidate Successful episodes                      268   
                                  Successful episodes using Shell          204   
Sparse Shell used                 All gold-annotated episodes              542   
                                  Successful episodes                      241   
       Non-gold Shell used        Successful episodes                      241   
       Shell workaround candidate Successful episodes                      241   
                                  Successful episodes using Shell          166   

                                                                   Rate (%)  
Cards  Metric                     Population                                 
Rich   Shell used                 All gold-annotated episodes          77.9  
                                  Successful episodes                  76.1  
       Non-gold Shell used        Successful episodes                  44.8  
       Shell workaround candidate Successful episodes                  20.1  
                                  Successful episodes using Shell      26.5  
Sparse Shell used                 All gold-annotated episodes          68.5  
                                  Successful episodes                  68.9  
       Non-gold Shell used        Successful episodes                  38.6  
       Shell workaround candidate Successful episodes                  17.4  
                                  Successful episodes using Shell      25.3

## Dynamic-Pool Bridge

This paired training-window comparison shows whether Shell usage shifts after `env_explorer` becomes available.


In [6]:
usage_shift = agents.agent_usage_shift_table(
    "officebench", fold_average=True
)
display(usage_shift[usage_shift["Agent"].isin(["shell", "env_explorer"])].set_index("Agent"))


,Task usage (%) Original,Task usage (%) Dynamic,Delta task usage (pp),Agent-use share (%) Original,Agent-use share (%) Dynamic,Delta agent-use share (pp)
Agent,,,,,,
shell,77.6,70.6,-7.0,35.9,26.8,-9.1
env_explorer,0.0,32.0,32.0,0.0,12.2,12.2
